In [1]:
### ARTS as the line-by-line code
import os
# set global variables BEFORE importing ARTS
os.environ['ARTS_DATA_PATH'] = '../arts-cat-data-2.6.14:../arts-xml-data-2.6.14'

import pyarts
from pyarts.arts import convert
# the fluxsim module
import FluxSimulator as fsm
import seaborn as sns


# helper packages
import numpy as np
import xarray as xr
from scipy.constants import speed_of_light as c
import matplotlib.pyplot as plt

import typhon
import typhon.constants as tpc
import pandas as pd

from scipy.optimize import minimize

from cycler import cycler

In [2]:
ddq_sw = xr.open_dataset('/homedata/pczarnec/ddq-implementation-sprint/DDQ configurations/DDQ_SW_source.h5', engine = "netcdf4").compute()
ddq_lw = xr.open_dataset('/homedata/pczarnec/ddq-implementation-sprint/DDQ configurations/DDQ_LW.h5', engine = "netcdf4").compute()

getfattr: /homedata/pczarnec/ddq-implementation-sprint/DDQ: No such file or directory
getfattr: configurations/DDQ_SW_source.h5: No such file or directory
getfattr: /homedata/pczarnec/ddq-implementation-sprint/DDQ: No such file or directory
getfattr: configurations/DDQ_LW.h5: No such file or directory


In [17]:
f_grid = convert.kaycm2freq(ddq_sw.S.data)

ws = pyarts.Workspace()
ws.f_grid = f_grid
ws.stokes_dim = 1

ws.rtp_pressure = 101325.0     # Pa  (1 atm)
ws.rtp_temperature = 273.15    # K   (0 C)

ws.gas_scattering_coefAirSimple()

data = np.asarray(ws.gas_scattering_coef.value.data)
beta_sca = data[0, 0, :, 0]  # [1/m], one value per frequency in f_grid

k_B = 1.380649e-23  # J/K
n_number_density = ws.rtp_pressure.value / (k_B * ws.rtp_temperature.value)  # [1/m^3]
 
sigma_sca = beta_sca / n_number_density  # [m^2] per molecule

In [20]:
ddq_sw = ddq_sw.assign({"Rayleigh_xsec" : (("S"), sigma_sca)})
ddq_sw

<xarray.Dataset> Size: 2kB
Dimensions:        (S: 64)
Coordinates:
  * S              (S) float64 512B 2.144e+03 2.411e+03 ... 4.426e+04 4.839e+04
Data variables:
    W              (S) float64 512B 1.509e+03 1.244e+03 ... 0.01592 0.01849
    solar_source   (S) float64 512B 456.1 596.5 838.6 ... 9.108 9.925 2.085
    Rayleigh_xsec  (S) float64 512B 8.392e-35 1.343e-34 ... 2.008e-29 3.076e-29
Attributes:
    description:  64 optimized wavenumbers and weights, in the shortwave, opt...

In [23]:
ddq_sw.to_netcdf("../DDQ configurations/DDQ_SW_source_Rayleigh.h5")

In [79]:
### test xsec

profile = xr.open_dataset('rfmip_expt0.nc').isel(expt = 0).isel(layer = slice(None, None, -1), level = slice(None, None, -1))
profile

<xarray.Dataset> Size: 171kB
Dimensions:                 (site: 100, layer: 60, level: 61)
Dimensions without coordinates: site, layer, level
Data variables: (12/20)
    pressure_layer          (site, layer) float32 24kB ...
    pressure_level          (site, level) float32 24kB ...
    surface_emissivity      (site) float32 400B ...
    surface_albedo          (site) float32 400B ...
    solar_zenith_angle      (site) float32 400B ...
    total_solar_irradiance  (site) float32 400B ...
    ...                      ...
    CFC12                   float32 4B ...
    CO2                     float32 4B ...
    CFC11                   float32 4B ...
    N2O                     float32 4B ...
    CH4                     float32 4B ...
    N_per_m2_dry            (site, layer) float32 24kB ...
Attributes: (12/25)
    title:               Atmospheric conditions for off-line radiative transf...
    institution_id:      UColorado
    institution:         University of Colorado, Boulder, CO 80309, USA
    activity_id:         input4MIPs
    Conventions:         CF-1.6
    creation_date:       2019-03-20 16:07:21-0400
    ...                  ...
    nominal_resolution:  10 km
    target_mip:          RFMIP
    variable_id:         multiple
    grid_label:          none
    tracking_id:         hdl:21.14100/f379c294-f7bb-442d-bca7-64661d60780e
    license:             Atmospheric condition data for RFMIP produced by the...

In [102]:
atm_idx = 0
results_xrs = []

for atm_idx in range(len((profile.site.data))):
    H2O_level =  np.interp(profile.pressure_level.data[atm_idx, ::-1], profile.pressure_layer.data[atm_idx, ::-1], profile.H2O.data[atm_idx, ::-1])[::-1]
    O3_level = np.interp(profile.pressure_level.data[atm_idx, ::-1], profile.pressure_layer.data[atm_idx, ::-1], profile.O3.data[atm_idx, ::-1])[::-1]
    pressure = profile.pressure_level.data[atm_idx, :].copy()
    pressure[-1] = np.max([pressure[-1], 1])
    atm_field = fsm.generate_gridded_field_from_profiles(
        pressure.tolist(), # pressure, in units Pa; must be a list
        profile.temperature_level.data[atm_idx, :], # temerpature, in units K
        gases={"H2O": H2O_level,
            "CO2": profile.CO2.data, 
            "O3": O3_level,
            "CH4": profile.CH4.data,
            "N2O": profile.N2O.data,
            "O2": profile.O2.data,
            "N2": profile.N2.data,
            "CFC12" : profile.CFC12.data,
            "CFC11" : profile.CFC11.data
            },
        )

    tsurf = profile.surface_temperature.data[atm_idx]
    alt = atm_field[1][0][0][0]
    sza = profile.solar_zenith_angle.data[atm_idx]
    if (sza <= -90) or (sza >= 90):
        sza = 89.99

    geo = [sza, 0]
    sun = [1.49331100e11, 0.0, 0.0] # moved so that insolation matches LMDZ
    alb = [profile.surface_albedo.data[atm_idx]]

    FluxSimulator_SW = fsm.FluxSimulator("sw")
    FluxSimulator_SW.ws.disort_aux_vars = ["Layer optical thickness", "Single scattering albedo", "Asymmetry parameter"]

    FluxSimulator_SW.ws.f_grid = convert.kaycm2freq(ddq_sw.S.data)
    FluxSimulator_SW.set_paths(lut_path='/homedata/pczarnec/ddq-implementation-sprint/DDQ configurations/LUTs/DDQ_sw')
    FluxSimulator_SW.set_species(
        ["H2O, H2O-SelfContCKDMT400, H2O-ForeignContCKDMT400",
            "O2-*-1e12-1e99,O2-CIAfunCKDMT100",
            "N2, N2-CIAfunCKDMT252, N2-CIArotCKDMT252",
            "CH4",
            "CO2, CO2-CKDMT252",
            "N2O",
            "O3",
            "O3-XFIT",    
            ]
    )


    FluxSimulator_SW.emission = 0
    FluxSimulator_SW.gas_scattering = True

    FluxSimulator_SW.sunspectrumtype = '/homedata/pczarnec/arts-xml-data-2.6.14/star/Sun/solar_spectrum_July_2008.xml'
    FluxSimulator_SW.set_sun()
    FluxSimulator_SW.scale_sun_to_specific_TSI_at_TOA(profile.total_solar_irradiance.data[atm_idx], geo[0], geo[1], atm_field[1][-1][0][0])
    FluxSimulator_SW.print_config()

    result = FluxSimulator_SW.flux_simulator_single_profile(
        atm_field,
        tsurf,
        alt,
        alb,
        geo,
    )

    atm_xr = xr.Dataset(
        data_vars = dict(
            layer_optical_thickness = (["S", "layer"], result["aux_var_clearsky"][0]),
            single_scattering_albedo = (["S", "layer"], result["aux_var_clearsky"][1]),
            asymmetry_parameter = (["S", "layer"], result["aux_var_clearsky"][2]),
            flux_clearsky_up =  (["S", "level"], result["spectral_flux_clearsky_up"]),
            flux_clearsky_down =  (["S", "level"], result["spectral_flux_clearsky_down"]),
        ),
        coords = dict(
            S = ddq_sw.S.data,
            layer = profile.layer,
            level = profile.level,
        )
    )

    results_xrs.append(atm_xr)


setup_name:  sw
species:  ['H2O, H2O-SelfContCKDMT400, H2O-ForeignContCKDMT400', 'O2-*-1e12-1e99,O2-CIAfunCKDMT100', 'N2, N2-CIAfunCKDMT252, N2-CIArotCKDMT252', 'CH4', 'CO2, CO2-CKDMT252', 'N2O', 'O3', 'O3-XFIT']
Cp:  1005.7
nstreams:  10
emission:  0
quadrature_weights:  
allsky:  False
gas_scattering:  True
sunspectrumtype:  /homedata/pczarnec/arts-xml-data-2.6.14/star/Sun/solar_spectrum_July_2008.xml
basename_scatterer:  /homedata/pczarnec/pyarts-fluxes/src/FluxSimulator/../../scattering_data
lut_path:  /homedata/pczarnec/ddq-implementation-sprint/DDQ configurations/LUTs/DDQ_sw
lutname_fullpath:  /homedata/pczarnec/ddq-implementation-sprint/DDQ configurations/LUTs/DDQ_sw/LUT.xml
setting up absorption...

...using stored LUT

starting calculation...

setup_name:  sw
species:  ['H2O, H2O-SelfContCKDMT400, H2O-ForeignContCKDMT400', 'O2-*-1e12-1e99,O2-CIAfunCKDMT100', 'N2, N2-CIAfunCKDMT252, N2-CIArotCKDMT252', 'CH4', 'CO2, CO2-CKDMT252', 'N2O', 'O3', 'O3-XFIT']
Cp:  1005.7
nstreams:  1

Start disort
Start disort


setup_name:  sw
species:  ['H2O, H2O-SelfContCKDMT400, H2O-ForeignContCKDMT400', 'O2-*-1e12-1e99,O2-CIAfunCKDMT100', 'N2, N2-CIAfunCKDMT252, N2-CIArotCKDMT252', 'CH4', 'CO2, CO2-CKDMT252', 'N2O', 'O3', 'O3-XFIT']
Cp:  1005.7
nstreams:  10
emission:  0
quadrature_weights:  
allsky:  False
gas_scattering:  True
sunspectrumtype:  /homedata/pczarnec/arts-xml-data-2.6.14/star/Sun/solar_spectrum_July_2008.xml
basename_scatterer:  /homedata/pczarnec/pyarts-fluxes/src/FluxSimulator/../../scattering_data
lut_path:  /homedata/pczarnec/ddq-implementation-sprint/DDQ configurations/LUTs/DDQ_sw
lutname_fullpath:  /homedata/pczarnec/ddq-implementation-sprint/DDQ configurations/LUTs/DDQ_sw/LUT.xml
setting up absorption...

...using stored LUT

starting calculation...

setup_name:  sw
species:  ['H2O, H2O-SelfContCKDMT400, H2O-ForeignContCKDMT400', 'O2-*-1e12-1e99,O2-CIAfunCKDMT100', 'N2, N2-CIAfunCKDMT252, N2-CIArotCKDMT252', 'CH4', 'CO2, CO2-CKDMT252', 'N2O', 'O3', 'O3-XFIT']
Cp:  1005.7
nstreams:  1

Start disort
Start disort


setup_name:  sw
species:  ['H2O, H2O-SelfContCKDMT400, H2O-ForeignContCKDMT400', 'O2-*-1e12-1e99,O2-CIAfunCKDMT100', 'N2, N2-CIAfunCKDMT252, N2-CIArotCKDMT252', 'CH4', 'CO2, CO2-CKDMT252', 'N2O', 'O3', 'O3-XFIT']
Cp:  1005.7
nstreams:  10
emission:  0
quadrature_weights:  
allsky:  False
gas_scattering:  True
sunspectrumtype:  /homedata/pczarnec/arts-xml-data-2.6.14/star/Sun/solar_spectrum_July_2008.xml
basename_scatterer:  /homedata/pczarnec/pyarts-fluxes/src/FluxSimulator/../../scattering_data
lut_path:  /homedata/pczarnec/ddq-implementation-sprint/DDQ configurations/LUTs/DDQ_sw
lutname_fullpath:  /homedata/pczarnec/ddq-implementation-sprint/DDQ configurations/LUTs/DDQ_sw/LUT.xml
setting up absorption...

...using stored LUT

starting calculation...

setup_name:  sw
species:  ['H2O, H2O-SelfContCKDMT400, H2O-ForeignContCKDMT400', 'O2-*-1e12-1e99,O2-CIAfunCKDMT100', 'N2, N2-CIAfunCKDMT252, N2-CIArotCKDMT252', 'CH4', 'CO2, CO2-CKDMT252', 'N2O', 'O3', 'O3-XFIT']
Cp:  1005.7
nstreams:  1

...

Start disort
Start disort


setup_name:  sw
species:  ['H2O, H2O-SelfContCKDMT400, H2O-ForeignContCKDMT400', 'O2-*-1e12-1e99,O2-CIAfunCKDMT100', 'N2, N2-CIAfunCKDMT252, N2-CIArotCKDMT252', 'CH4', 'CO2, CO2-CKDMT252', 'N2O', 'O3', 'O3-XFIT']
Cp:  1005.7
nstreams:  10
emission:  0
quadrature_weights:  
allsky:  False
gas_scattering:  True
sunspectrumtype:  /homedata/pczarnec/arts-xml-data-2.6.14/star/Sun/solar_spectrum_July_2008.xml
basename_scatterer:  /homedata/pczarnec/pyarts-fluxes/src/FluxSimulator/../../scattering_data
lut_path:  /homedata/pczarnec/ddq-implementation-sprint/DDQ configurations/LUTs/DDQ_sw
lutname_fullpath:  /homedata/pczarnec/ddq-implementation-sprint/DDQ configurations/LUTs/DDQ_sw/LUT.xml
setting up absorption...

...using stored LUT

starting calculation...

setup_name:  sw
species:  ['H2O, H2O-SelfContCKDMT400, H2O-ForeignContCKDMT400', 'O2-*-1e12-1e99,O2-CIAfunCKDMT100', 'N2, N2-CIAfunCKDMT252, N2-CIArotCKDMT252', 'CH4', 'CO2, CO2-CKDMT252', 'N2O', 'O3', 'O3-XFIT']
Cp:  1005.7
nstreams:  1

Start disort


...using stored LUT

starting calculation...



Start disort


In [103]:
final = xr.concat(results_xrs, 'site')

In [104]:
final

<xarray.Dataset> Size: 15MB
Dimensions:                   (site: 100, S: 64, layer: 60, level: 61)
Coordinates:
  * S                         (S) float64 512B 2.144e+03 2.411e+03 ... 4.839e+04
  * layer                     (layer) int64 480B 0 1 2 3 4 5 ... 55 56 57 58 59
  * level                     (level) int64 488B 0 1 2 3 4 5 ... 56 57 58 59 60
Dimensions without coordinates: site
Data variables:
    layer_optical_thickness   (site, S, layer) float64 3MB 0.001033 ... 0.002016
    single_scattering_albedo  (site, S, layer) float64 3MB 2.794e-05 ... 1.0
    asymmetry_parameter       (site, S, layer) float64 3MB 0.0 0.0 ... 0.0 0.0
    flux_clearsky_up          (site, S, level) float64 3MB 2.735e-14 ... 6.74...
    flux_clearsky_down        (site, S, level) float64 3MB -1.57e-13 ... -7.9...

In [105]:
final.to_netcdf('RFMIP_at_DDQ_SW.nc')